# Prueba de la API de Gemini para A5

Notebook de referencia para configurar `A5_Modelo_Comercial_G15.ipynb` con el modelo de Google. Comprueba la clave, el endpoint y el formato de respuesta, y estima el costo y el tiempo de la corrida completa sobre el Test.

## Decisión (15-nov-2026)

A5 evalúa **dos modelos comerciales**: uno de **OpenAI**, de gama económica, que prueba y configura el compañero de A5 por separado, y uno de **Google**, que es el que cubre este notebook:

| Modelo | ID exacto | USD por 1 M tokens (entrada / salida) | Límites (Tier 1) |
|---|---|---|---|
| Gemini 3.5 Flash-Lite | `gemini-3.5-flash-lite` | 0,30 / 2,50 | 4.000 por minuto · 150.000 por día |

- **Por qué este modelo:** gama económica comparable a la del modelo de OpenAI, versión estable, límites holgados y la latencia más baja en la prueba.
- **Descartados:** la familia 2.5 (`gemini-2.5-flash-lite`, `-flash`, `-pro`) aparece en la lista pero responde *404 no longer available to new users*; `gemini-3.1-flash-lite` funcionó, pero dio errores 503 por alta demanda y fue unas 8 veces más lento; `gemini-3-flash-preview` es una versión *preview* que puede cambiar; Gemini 3.8 Flash cuesta ≈2,5 veces más y tiene 10.000 llamadas por día; Gemma 4 fue uno de los jueces del pre-etiquetado.
- **Usar siempre el ID exacto**, no alias como `gemini-flash-lite-latest`, que Google puede reasignar.
- Precios: <https://ai.google.dev/gemini-api/docs/pricing>, consultados el 15-nov-2026.

## Resultados de la prueba (15-nov-2026)

Con los prompts reales de A5 (caso y ejemplos inventados, no del Test), `gemini-3.5-flash-lite` devolvió **8 de 8 respuestas válidas** en las cuatro condiciones, todas con la función esperada:

| Condición | Tokens de entrada | Tokens de salida | Latencia |
|---|---:|---:|---:|
| genérico, zero-shot | 788 | ~109 | ~1,0 s |
| genérico, few-shot | 3.975 | 65 | ~1,2 s |
| específico, zero-shot | 3.513 | 83–109 | ~1,3 s |
| específico, few-shot | 6.700 | 65 | ~1,7 s |

- **Costo proyectado** de la corrida completa (2.968 × 4 = 11.872 llamadas): **USD 15,9**; **USD 19,1** con 20 % de margen por reintentos y piloto; ≈USD 11 si el caché implícito actúa como en la prueba.
- **Tiempo:** ~4 h en secuencia.
- **Endpoint compatible con OpenAI:** acepta `max_completion_tokens` y `max_tokens`, y `response_format: json_object` devuelve JSON válido. Con `reasoning_effort: low` no hubo tokens de razonamiento cobrados.

## Cómo ejecutarlo

1. **Clave** (nunca en el código ni en la salida): en Colab, 🔑 *Secrets* → `A5_API_KEY` con *Notebook access*; en local, archivo `A5/.env` con la línea `A5_API_KEY=...` (ignorado por git).
2. **Kernel:** en local, *Python (A5 .venv)*; solo usa la biblioteca estándar más `a5_runtime.py`.
3. Ejecutar todo en orden. Son unas 30 llamadas, con un costo de centavos.

| Sección | Qué hace |
|---|---|
| 0 | Carga la clave y define las utilidades HTTP |
| 1 | Lista los modelos de texto de Gemini visibles para la clave |
| 2 | Fija el modelo de A5 y su precio |
| 3 | Llamada nativa mínima |
| 4 | Endpoint compatible con OpenAI (el del notebook de A5) y parámetro de salida aceptado |
| 5 | Prepara las entradas de A5 con los prompts reales y casos inventados |
| 6 | Mide tokens, latencia y validez en las cuatro condiciones |
| 7 | Proyecta costo y tiempo de la corrida completa |
| 8 | Imprime la configuración para copiar en el notebook de A5 |

## 0. Configuración: clave y utilidades

In [1]:
import os, json, sys, time, urllib.request, urllib.error
from datetime import datetime, timezone
from pathlib import Path

KEY_NAME = "A5_API_KEY"
BASE = "https://generativelanguage.googleapis.com/v1beta"
OPENAI_URL = f"{BASE}/openai/chat/completions"


def load_key(name=KEY_NAME):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip(), "Colab Secrets"
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip(), "variable de entorno"
    for path in (Path.cwd() / ".env", Path.cwd() / "A5" / ".env"):
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith(name + "="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'"), str(path)
    raise RuntimeError(f"No se encontró {name} en Colab Secrets, el entorno ni A5/.env")


API_KEY, KEY_SOURCE = load_key()
NATIVE = {"x-goog-api-key": API_KEY}
OPENAI = {"Authorization": f"Bearer {API_KEY}"}
print(f"Clave cargada desde {KEY_SOURCE} ({len(API_KEY)} caracteres).")

Clave cargada desde /Users/germanrodriguez/Desktop/Code/GMRM/MAIA/Proyecto_Grado_MAIA/A5/.env (53 caracteres).


In [2]:
def call(method, url, body=None, headers=None, timeout=120):
    data = json.dumps(body).encode() if body is not None else None
    request = urllib.request.Request(url, data=data, method=method,
                                     headers={"Content-Type": "application/json", **(headers or {})})
    start = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status, text = response.status, response.read().decode()
    except urllib.error.HTTPError as error:
        status, text = error.code, error.read().decode(errors="replace")
    elapsed = time.perf_counter() - start
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = {"raw": text[:500]}
    return status, payload, elapsed


def show_error(status, payload):
    if isinstance(payload, list) and payload:  # el endpoint compatible con OpenAI puede devolver una lista
        payload = payload[0]
    detail = payload.get("error", payload) if isinstance(payload, dict) else payload
    print(f"HTTP {status}:", json.dumps(detail, ensure_ascii=False, indent=1)[:800])

## 1. Modelos disponibles para esta clave

Solo modelos de texto de Gemini (se omiten los de imagen, audio, agentes y robótica).

In [3]:
SKIP = ("tts", "image", "banana", "lyria", "robotics", "transcribe", "computer-use", "customtools",
        "deep-research", "antigravity", "omni")
models, token = [], None
while True:
    url = f"{BASE}/models?pageSize=200" + (f"&pageToken={token}" if token else "")
    status, payload, _ = call("GET", url, headers=NATIVE)
    if status != 200:
        show_error(status, payload)
        break
    models += payload.get("models", [])
    token = payload.get("nextPageToken")
    if not token:
        break

generative = [m for m in models if "generateContent" in m.get("supportedGenerationMethods", [])]
available = {m["name"].removeprefix("models/") for m in generative}
text_models = sorted((m for m in generative if m["name"].startswith("models/gemini")
                      and not any(s in m["name"] for s in SKIP)), key=lambda m: m["name"])
print(f"Modelos con generateContent: {len(generative)} · modelos de texto Gemini: {len(text_models)}\n")
print(f"{'ID':32} {'Nombre':30} {'entrada':>9} {'salida':>7}")
for m in text_models:
    model_id = m["name"].removeprefix("models/")
    print(f"{model_id:32} {m.get('displayName', '')[:30]:30} {m.get('inputTokenLimit', ''):>9} "
          f"{m.get('outputTokenLimit', ''):>7}")
print("\nNota: la familia 2.5 aparece en la lista pero no está disponible para proyectos nuevos (HTTP 404).")

Modelos con generateContent: 49 · modelos de texto Gemini: 18

ID                               Nombre                           entrada  salida
gemini-2.5-flash                 Gemini 2.5 Flash                 1048576   65536
gemini-2.5-flash-lite            Gemini 2.5 Flash-Lite            1048576   65536
gemini-2.5-pro                   Gemini 2.5 Pro                   1048576   65536
gemini-3-flash-preview           Gemini 3 Flash Preview           1048576   65536
gemini-3.1-flash-lite            Gemini 3.1 Flash Lite            1048576   65536
gemini-3.1-flash-lite-preview    Gemini 3.1 Flash Lite Preview    1048576   65536
gemini-3.1-pro-preview           Gemini 3.1 Pro Preview           1048576   65536
gemini-3.5-flash                 Gemini 3.5 Flash                 1048576   65536
gemini-3.5-flash-lite            Gemini 3.5 Flash Lite            1048576   65536
gemini-3.6-flash                 Gemini 3.6 Flash                 1048576   65536
gemini-3.7-flash                 Ge

## 2. Modelo de A5

Modelo elegido y su precio oficial (nivel estándar pagado).

In [4]:
PRICES = {  # USD por 1 M tokens; ai.google.dev/gemini-api/docs/pricing, 15-nov-2026
    "gemini-3.5-flash-lite": {"input": 0.30, "cached": 0.03, "output": 2.50},
}
MODELS = list(PRICES)  # para comparar otro modelo de Google, añádelo aquí con su precio
missing = [m for m in MODELS if m not in available]
assert not missing, f"No disponibles para esta clave: {missing}"
print("Modelo de A5:", MODELS)

Modelo de A5: ['gemini-3.5-flash-lite']


## 3. Llamada nativa mínima (`generateContent`)

In [5]:
for model in MODELS:
    body = {"contents": [{"parts": [{"text": "Responde solo con la palabra OK."}]}],
            "generationConfig": {"maxOutputTokens": 256}}
    status, payload, elapsed = call("POST", f"{BASE}/models/{model}:generateContent", body, NATIVE)
    if status == 200:
        parts = payload["candidates"][0].get("content", {}).get("parts", [])
        text = "".join(p.get("text", "") for p in parts).strip()[:50]
        print(f"{model:24} respuesta={text!r} · {elapsed:.2f} s · tokens={payload.get('usageMetadata', {}).get('totalTokenCount')}")
    else:
        print(model, end=": ")
        show_error(status, payload)

gemini-3.5-flash-lite    respuesta='OK' · 0.70 s · tokens=9


## 4. Endpoint compatible con OpenAI (el que usa el notebook de A5)

Comprueba qué parámetro de límite de salida acepta cada modelo y si `response_format: json_object` devuelve JSON válido.

In [6]:
param_ok = {}
for model in MODELS:
    param_ok[model] = {}
    for param in ("max_completion_tokens", "max_tokens"):
        body = {"model": model,
                "messages": [{"role": "user", "content": 'Devuelve exactamente este JSON: {"ok": true}'}],
                "response_format": {"type": "json_object"}, param: 1024}
        status, payload, elapsed = call("POST", OPENAI_URL, body, OPENAI)
        if status == 200:
            content = payload["choices"][0]["message"].get("content") or ""
            try:
                valid = json.loads(content) == {"ok": True}
            except json.JSONDecodeError:
                valid = False
            param_ok[model][param] = valid
            print(f"{model:24} {param:22} HTTP 200 · JSON correcto={valid} · {elapsed:.2f} s")
        else:
            param_ok[model][param] = False
            print(f"{model:24} {param:22}", end=" ")
            show_error(status, payload)

gemini-3.5-flash-lite    max_completion_tokens  HTTP 200 · JSON correcto=True · 0.81 s


gemini-3.5-flash-lite    max_tokens             HTTP 200 · JSON correcto=True · 0.70 s


## 5. Preparar las entradas de A5

Construye los mensajes con `a5_runtime.build_messages` y `prompt_v11_1_snapshot.json`, en las **cuatro condiciones** (genérico/específico × zero/few-shot). El caso y los nueve ejemplos son **inventados**, con una longitud similar a la entrada real de SciBERT base (~380 tokens en promedio en el Test).

In [7]:
A5_DIR = Path.cwd() if (Path.cwd() / "a5_runtime.py").exists() else Path.cwd() / "A5"
sys.path.insert(0, str(A5_DIR))
import a5_runtime as a5

snapshot = json.loads((A5_DIR / "prompt_v11_1_snapshot.json").read_text(encoding="utf-8"))
filler = ("The proposed approach is evaluated on several public benchmarks and compared against strong baselines, "
          "with ablations on each component and an analysis of failure cases under distribution shift. ")
context = ("Recent object detectors reach high accuracy on standard benchmarks but degrade under occlusion. "
           "Following prior work, we adopt the training schedule and the optimizer proposed in [[TARGET]] "
           "without modification, and we report results averaged over three seeds. " + filler * 3)
abstract = ("We introduce an algorithm for first-order gradient-based optimization of stochastic objective functions, "
            "based on adaptive estimates of lower-order moments. " + filler * 4)
case = {"inst_key": "synthetic-case", "context": context, "target_sentence": context.split(". ")[1] + ".",
        "cited_title": "A Method for Stochastic Optimization", "cited_abstract": abstract}
demo = a5.synthetic_demo()
examples = [dict(ex, context=ex["context"] + " " + filler * 3, cited_abstract=ex["cited_abstract"] + " " + filler * 4)
            for ex in demo["examples"]]
conditions = [{"style": s, "shot": k, "representation": "base"} for s in ("generic", "specific") for k in ("zero", "few")]
print("Condiciones:", [f"{c['style']}-{c['shot']}" for c in conditions], "· respuesta esperada: Application")

Condiciones: ['generic-zero', 'generic-few', 'specific-zero', 'specific-few'] · respuesta esperada: Application


## 6. Medición de tokens y validez en las cuatro condiciones

Cada modelo × condición se llama **dos veces** (la segunda muestra si el caché implícito abarata la llamada). Usa `reasoning_effort: low` para limitar los tokens de razonamiento, que se cobran como salida. La respuesta se valida con `a5_runtime.validate_scores`, el mismo validador del notebook de A5 (nueve funciones, puntajes de 0 a 1 que suman 1).

In [8]:
REASONING_EFFORT = "low"  # None para omitirlo
measurements = []
for model in MODELS:
    param = next((p for p, ok in param_ok[model].items() if ok), "max_completion_tokens")
    for condition in conditions:
        messages = a5.build_messages(case, condition, snapshot, examples, demo["example_gold"])
        for attempt in (1, 2):
            body = {"model": model, "messages": messages, "response_format": {"type": "json_object"}, param: 2048}
            if REASONING_EFFORT:
                body["reasoning_effort"] = REASONING_EFFORT
            status, payload, elapsed = call("POST", OPENAI_URL, body, OPENAI)
            if status == 400 and REASONING_EFFORT:
                body.pop("reasoning_effort")
                status, payload, elapsed = call("POST", OPENAI_URL, body, OPENAI)
            name = f"{condition['style']}-{condition['shot']}"
            if status != 200:
                print(model, name, attempt, end=": ")
                show_error(status, payload)
                continue
            usage = payload.get("usage", {})
            parsed = a5.validate_scores(payload["choices"][0]["message"].get("content") or "")
            measurements.append({
                "model": model, "condition": name, "attempt": attempt, "input": usage.get("prompt_tokens", 0),
                "cached": (usage.get("prompt_tokens_details") or {}).get("cached_tokens", 0) or 0,
                "output": usage.get("completion_tokens", 0),
                "reasoning": (usage.get("completion_tokens_details") or {}).get("reasoning_tokens", 0) or 0,
                "valid": parsed["valid"], "pred": parsed["predicted_label"], "error": parsed["error"],
                "seconds": round(elapsed, 2)})

print(f"{'modelo':22} {'condición':15} {'int':>3} {'entrada':>8} {'caché':>6} {'salida':>7} {'razon.':>6} {'válida':>6} {'s':>6}  predicción / error")
for m in measurements:
    print(f"{m['model']:22} {m['condition']:15} {m['attempt']:>3} {m['input']:>8} {m['cached']:>6} {m['output']:>7} "
          f"{m['reasoning']:>6} {str(m['valid']):>6} {m['seconds']:>6}  {m['pred'] if m['valid'] else m['error']}")

modelo                 condición       int  entrada  caché  salida razon. válida      s  predicción / error
gemini-3.5-flash-lite  generic-zero      1      788      0     109      0   True   1.02  Application
gemini-3.5-flash-lite  generic-zero      2      788      0     109      0   True   1.04  Application
gemini-3.5-flash-lite  generic-few       1     3975   1961      65      0   True   1.27  Application
gemini-3.5-flash-lite  generic-few       2     3975   1961      65      0   True   1.07  Application
gemini-3.5-flash-lite  specific-zero     1     3513      0     109      0   True   1.18  Application
gemini-3.5-flash-lite  specific-zero     2     3513      0      83      0   True   1.38  Application
gemini-3.5-flash-lite  specific-few      1     6700   3991      65      0   True   1.43  Application
gemini-3.5-flash-lite  specific-few      2     6700   3991      65      0   True    1.9  Application


## 7. Costo proyectado de la corrida completa

2.968 instancias del Test × 4 condiciones = 11.872 llamadas. *Sin caché* usa la primera llamada de cada condición; *con caché*, la segunda. Sumar ~20 % por reintentos y el piloto.

In [9]:
N_TEST = 2968
rows = []
for model in MODELS:
    price = PRICES[model]
    totals = {}
    for use_cache in (False, True):
        total = 0.0
        for condition in conditions:
            name = f"{condition['style']}-{condition['shot']}"
            ms = [m for m in measurements if m["model"] == model and m["condition"] == name]
            if not ms:
                continue
            m = ms[-1] if use_cache else ms[0]
            cached = m["cached"] if use_cache else 0
            total += ((m["input"] - cached) * price["input"] + cached * price["cached"] + m["output"] * price["output"]) / 1e6
        totals[use_cache] = total * N_TEST
    seconds = sum(m["seconds"] for m in measurements if m["model"] == model and m["attempt"] == 1)
    rows.append((model, totals[False], totals[True], totals[False] * 1.2, seconds / len(conditions)))

print(f"{'modelo':24} {'sin caché':>10} {'con caché':>10} {'+20 %':>8} {'s/llamada':>10}")
for model, plain, cached, margin, sec in rows:
    print(f"{model:24} {plain:>9.2f}$ {cached:>9.2f}$ {margin:>7.2f}$ {sec:>10.1f}")
print(f"{'TOTAL':24} {sum(r[1] for r in rows):>9.2f}$ {sum(r[2] for r in rows):>9.2f}$ {sum(r[3] for r in rows):>7.2f}$")
for model, *_, sec in rows:
    print(f"Tiempo secuencial estimado para {model}: {sec * N_TEST * len(conditions) / 3600:.1f} h")

modelo                    sin caché  con caché    +20 %  s/llamada
gemini-3.5-flash-lite        15.92$     10.95$   19.10$        1.2
TOTAL                        15.92$     10.95$   19.10$
Tiempo secuencial estimado para gemini-3.5-flash-lite: 4.0 h


## 8. Configuración para `A5_Modelo_Comercial_G15.ipynb`

Valores comprobados en este notebook. Copiar en la sección 4 del notebook de A5 para la corrida de Gemini; la del modelo de OpenAI usa su propio endpoint, clave y precios.

**Avisos para la corrida real:**
- El notebook de A5 llama **en secuencia** (`REQUEST_INTERVAL_SECONDS = 0.5`). Con las latencias medidas, cada modelo puede tardar muchas horas: arrancar con tiempo o paralelizar sin superar 4.000 RPM.
- Antes del Test, correr el piloto de desarrollo (`SPLIT="dev"`, 30–50 casos) y revisar el costo real en AI Studio → *Spend*.
- La clave va en Colab Secrets como `A5_API_KEY`; nunca en el código.

In [10]:
for model in MODELS:
    param = next((p for p, ok in param_ok[model].items() if ok), None)
    ok = [m["valid"] for m in measurements if m["model"] == model]
    print(f"# --- {model}: {sum(ok)}/{len(ok)} respuestas válidas en la prueba ({datetime.now(timezone.utc):%Y-%m-%d}) ---")
    print(f'''ENABLE_API = True
API_ENDPOINT = "{OPENAI_URL}"
MODEL_ID = "{model}"
OUTPUT_TOKEN_PARAMETER = "{param}"
RESPONSE_FORMAT = "json_object"
EXTRA_BODY = {{"reasoning_effort": "{REASONING_EFFORT}"}}
INPUT_USD_PER_MILLION = {PRICES[model]["input"]}
OUTPUT_USD_PER_MILLION = {PRICES[model]["output"]}
PRICING_VERIFIED = True
PRICING_SOURCE = "https://ai.google.dev/gemini-api/docs/pricing"
PRICING_DATE = "2026-11-15"
RESERVE_PER_ATTEMPT_USD = {round((7000 * PRICES[model]["input"] + 2048 * PRICES[model]["output"]) / 1e6, 4)}  # peor caso
TIMEOUT_SECONDS = 120
''')

# --- gemini-3.5-flash-lite: 8/8 respuestas válidas en la prueba (2026-11-15) ---
ENABLE_API = True
API_ENDPOINT = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL_ID = "gemini-3.5-flash-lite"
OUTPUT_TOKEN_PARAMETER = "max_completion_tokens"
RESPONSE_FORMAT = "json_object"
EXTRA_BODY = {"reasoning_effort": "low"}
INPUT_USD_PER_MILLION = 0.3
OUTPUT_USD_PER_MILLION = 2.5
PRICING_VERIFIED = True
PRICING_SOURCE = "https://ai.google.dev/gemini-api/docs/pricing"
PRICING_DATE = "2026-11-15"
RESERVE_PER_ATTEMPT_USD = 0.0072  # peor caso
TIMEOUT_SECONDS = 120

